# 02 — CDC PLACES morbidity (Ohio counties)

**Question:** What is the prevalence of cardiovascular conditions in each Ohio county, and does this data link cleanly to the CDC WONDER mortality data?

**Data:** CDC PLACES: Local Data for Better Health, County Data, 2025 release (dataset `swc5-untb`). Model-based estimates, mostly from 2023 BRFSS survey data.
- **Measures:** coronary heart disease (`CHD`), stroke (`STROKE`), high blood pressure (`BPHIGH`)
- **Estimate type:** age-adjusted prevalence (`AgeAdjPrv`), % of adults 18+
- **Geography:** 88 Ohio counties, keyed by county FIPS code (`locationid`)

**Output:** one row per county with the three prevalence measures, checked against the 88 counties in the WONDER mortality data.

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")

# CDC's data API, filtered to Ohio. Ohio has ~88 counties x 40 measures x 2 estimate types,
# so raise the default 1,000-row limit or you'll silently get a partial download.
url = "https://data.cdc.gov/resource/swc5-untb.csv?stateabbr=OH&$limit=10000"

places_raw = pd.read_csv(url, dtype={"locationid": str})
places_raw.to_csv(RAW / "places_2025_county_OH.csv", index=False)
places_raw.shape

(7040, 22)

In [4]:
print(places_raw.shape)
print(places_raw["measureid"].unique())
print(places_raw["datavaluetypeid"].unique())
places_raw[["locationid", "locationname", "measureid", "datavaluetypeid", "data_value"]].head(10)

(7040, 22)
<StringArray>
[     'OBESITY',       'STROKE',        'BINGE',   'DEPRESSION',
    'ARTHRITIS',   'DISABILITY',    'COGNITION',  'SHUTUTILITY',
     'CSMOKING', 'COLON_SCREEN',     'MOBILITY',      'CHECKUP',
          'LPA',   'HOUSINSECU',        'PHLTH',        'BPMED',
       'CANCER',     'HIGHCHOL',      'CASTHMA',     'LACKTRPT',
       'BPHIGH',    'TEETHLOST',          'CHD',        'MHLTH',
         'COPD',        'SLEEP',      'HEARING',      'ACCESS2',
       'VISION',     'SELFCARE',    'INDEPLIVE',   'FOODINSECU',
     'MAMMOUSE',    'FOODSTAMP',   'CHOLSCREEN',   'LONELINESS',
     'DIABETES',        'GHLTH',   'EMOTIONSPT',       'DENTAL']
Length: 40, dtype: str
<StringArray>
['CrdPrv', 'AgeAdjPrv']
Length: 2, dtype: str


,locationid,locationname,measureid,datavaluetypeid,data_value
0,39015,Brown,OBESITY,CrdPrv,41.6
1,39017,Butler,OBESITY,CrdPrv,39.0
2,39017,Butler,STROKE,CrdPrv,3.3
3,39023,Clark,STROKE,CrdPrv,4.2
4,39051,Fulton,BINGE,CrdPrv,16.2
5,39071,Highland,OBESITY,CrdPrv,41.3
6,39075,Holmes,DEPRESSION,CrdPrv,29.3
7,39079,Jackson,ARTHRITIS,CrdPrv,33.5
8,39095,Lucas,OBESITY,CrdPrv,36.1
9,39097,Madison,DEPRESSION,AgeAdjPrv,25.2


In [ ]:
#Filter to just the measures we care about, and just the age-adjusted prevalence estimates.

measures = ["CHD", "STROKE", "BPHIGH"]

keep = (places_raw["measureid"].isin(measures)) & (places_raw["datavaluetypeid"] == "AgeAdjPrv")
places = places_raw[keep]

places.shape

(264, 22)

In [6]:
places_wide = places.pivot(index="locationid", columns="measureid", values="data_value")
places_wide = places_wide.reset_index()   # turn locationid back into a normal column for merging
places_wide.columns.name = None           # remove the leftover "measureid" label above the columns
places_wide.head()

,locationid,BPHIGH,CHD,STROKE
0,39001,35.1,7.3,3.7
1,39003,34.3,6.3,3.2
2,39005,34.9,6.3,3.2
3,39007,35.9,7.1,3.6
4,39009,36.6,6.6,3.4


In [7]:
assert places_wide.shape[0] == 88, "Expected 88 counties in Ohio"
assert places_wide.isna().sum().sum() == 0, "Expected no missing values in the filtered data"

In [8]:
places_wide.describe()

,BPHIGH,CHD,STROKE
count,88.000000,88.000000,88.000000
mean,33.984091,6.172727,3.129545
std,2.016876,0.623252,0.331197
min,28.200000,4.400000,2.200000
25%,32.675000,5.700000,2.900000
50%,33.850000,6.200000,3.150000
75%,35.325000,6.600000,3.400000
max,39.300000,7.700000,3.900000


The goal is to confirm that every Ohio county appears in both datasets under the same FIPS code. Every later step (adding noise, covariates, then the model) joins tables on this code, so a mismatch caught now saves you from a silent error later.

In [9]:
def load_wonder(path):
    """Load a CDC WONDER export, keeping only real data rows."""
    df = pd.read_csv(path, sep="\t", dtype=str)
    df = df[df["Notes"].isna() & df["County Code"].notna()].drop(columns="Notes")
    for col in ["Deaths", "Population", "Crude Rate"]:
        df[col + "_num"] = pd.to_numeric(df[col], errors="coerce")
    return df

mort = load_wonder(RAW / "A_ohio_county_2018_2022.txt")

In [10]:
merged = mort.merge(
    places_wide,
    left_on="County Code",   # FIPS column in WONDER
    right_on="locationid",   # FIPS column in PLACES
    how="outer",             # keep every county from both sides, matched or not
    indicator=True,          # add a _merge column saying where each row came from
)
merged["_merge"].value_counts()

_merge
both          88
left_only      0
right_only     0
Name: count, dtype: int64

In [11]:
assert (merged["_merge"] == "both").all(), merged.loc[merged["_merge"] != "both", ["County", "locationname", "_merge"]]

In [12]:
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)
places_wide.to_csv(PROCESSED / "places_oh_county_ageadj.csv", index=False)

## Interpretation

### Findings
- All **88 Ohio counties** have age-adjusted PLACES estimates for CHD, stroke, and high blood pressure, with no missing values.
- All 88 counties **link cleanly** to the CDC WONDER mortality data by county FIPS code.
- Age-adjusted prevalence ranges across counties:
  - Coronary heart disease: 4.4% to 7.7%
  - Stroke: 2.2% to 3.9%
  - High blood pressure: 28.2% to 39.3%
- Across counties, CHD prevalence is moderately correlated with the CVD death rate (r = 0.53); stroke and high blood pressure show similar correlations (r = 0.52 each).
- The three PLACES measures are highly correlated with each other (r = 0.88–0.95), so they largely rank counties the same way.

### Caveats
- **Model-based estimates.** PLACES values are predicted from BRFSS survey data plus census demographics (age, sex, race/ethnicity), not measured directly in each county. Some county-to-county variation therefore comes from demographics I will also adjust for, which could weaken or distort an estimated noise effect.
- **Different years.** PLACES reflects mostly 2023, mortality covers 2018–2022, and the noise data will be from 2022/2023. This assumes county noise levels and disease burden change slowly over a few years.
- **Different adjustment.** PLACES is age-adjusted, but the WONDER mortality rates are crude, so the two outcomes aren't directly comparable as rates.
- **Not independent outcomes.** Because the PLACES measures are so highly correlated (likely partly because they share the same model inputs), associations found for all three should be read as one finding, not three. CHD will be the primary morbidity outcome.

### Next step
Use the cleaned table (`data/processed/places_oh_county_ageadj.csv`) as the morbidity outcome. Next, build county-level noise exposure from the BTS 2022/2023 noise map (notebook 03).